# 05 - U.S. candidate baseline modeling (training only)

The U.S. model comparison reuses the same algorithms and fold-fitted shared preprocessing utilities as 1C's global `05_modeling_v3`. It uses the **U.S. training portion of the original global split**. The U.S. test set is not opened here.

For now use conservative launch-time attributes only. Do not compare its ROC-AUC directly to historical global CV or 1A/1B test numbers because the cohorts/evaluation protocols differ.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
US_DIR = GLOBAL_DIR / "us_only"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from utils.preprocessing import make_pipeline, drop_leaks
CLEAN = US_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
IDS = US_DIR / "train_project_ids.csv"
for path in [CLEAN, IDS]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; run U.S. notebooks 02 and 03")
clean = pd.read_csv(CLEAN)
train_ids = pd.read_csv(IDS)
train_rows = clean.set_index("project id", drop=False).loc[train_ids["project id"]].copy().reset_index(drop=True)
y_train = train_ids["target"].astype(int).reset_index(drop=True)
assert np.array_equal(train_rows["status"].eq("successful").astype(int), y_train)
X_train = drop_leaks(train_rows.drop(columns=["status"]))
assert not {"pledged", "funded percentage", "updates", "comments", "backers"}.intersection(X_train.columns)
print("Training rows:", len(X_train), "success rate:", y_train.mean())

Training rows: 26945 success rate: 0.546149563926517


In [2]:
# Conservative policy: no title/reward information or unverified date features.
FEATURE_POLICY = dict(include_timing=False, include_flags=False,
                      include_reward=False, exclude_renamed=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {"roc_auc": "roc_auc", "accuracy": "accuracy", "f1": "f1",
           "average_precision": "average_precision", "balanced_accuracy": "balanced_accuracy"}
MODELS = {
    "dummy": (DummyClassifier(strategy="prior"), "onehot"),
    "logistic": (LogisticRegression(max_iter=2000), "onehot"),
    "knn": (KNeighborsClassifier(n_neighbors=25), "onehot"),
    "decision_tree": (DecisionTreeClassifier(max_depth=8, min_samples_leaf=40, random_state=42), "onehot"),
    "random_forest": (RandomForestClassifier(n_estimators=250, min_samples_leaf=5, n_jobs=1, random_state=42), "target"),
    "hist_gradient_boosting": (HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, random_state=42), "target"),
}
rows = []
for name, (estimator, categorical_strategy) in MODELS.items():
    pipeline = make_pipeline(estimator, categorical_strategy=categorical_strategy, **FEATURE_POLICY)
    scores = cross_validate(pipeline, X_train, y_train, cv=CV, scoring=SCORING,
                            n_jobs=1, error_score="raise")
    row = {"model": name}
    for metric in SCORING:
        row[metric] = scores[f"test_{metric}"].mean()
    row["roc_auc_fold_sd"] = scores["test_roc_auc"].std()
    rows.append(row)
results = pd.DataFrame(rows).set_index("model").sort_values("roc_auc", ascending=False)
display(results.round(4))

,roc_auc,accuracy,f1,average_precision,balanced_accuracy,roc_auc_fold_sd
model,,,,,,
hist_gradient_boosting,0.7235,0.6684,0.7156,0.7377,0.6587,0.0082
logistic,0.7199,0.6684,0.7111,0.7286,0.6604,0.0055
random_forest,0.7117,0.6583,0.7036,0.7261,0.6498,0.0065
knn,0.6990,0.6531,0.7073,0.7056,0.6415,0.0057
decision_tree,0.6867,0.6491,0.6958,0.6902,0.6403,0.0045
dummy,0.5000,0.5461,0.7065,0.5461,0.5000,0.0000


In [3]:
ax = results.drop(index="dummy")["roc_auc"].sort_values().plot.barh(figsize=(9, 4))
ax.set_xlabel("5-fold cross-validation ROC-AUC (U.S. candidate training only)")
ax.axvline(0.5, linestyle="--", color="gray")
plt.tight_layout()

## Handoff to `06` (with Zasha)

1. Resolve country/geographic-scope and launch-time field questions.
2. Decide whether to keep the conservative feature policy or enable any *verified* groups.
3. Tune the strongest baseline candidates using **only U.S. training rows**.
4. Select a model using cross-validation, then do one final U.S. holdout evaluation.
5. Because U.S. train/test IDs are subsets of global official IDs, a later global-vs-U.S. comparison can use **the same U.S. test IDs**, without generating a conflicting new split.

**Don't run 06 on test labels until model specification is frozen.** The existing shared `06` notebook remains unchanged for you and Zasha to build together.